# Agentic AI — Automated UI Test Generation

**Pipeline:** SRS document -> requirements -> crawl the live site -> generate a
Playwright pytest suite -> validate it -> loop back and fix until it passes.

Four agents wired with LangGraph:

| Agent | Job |
|---|---|
| Extractor | Reads the SRS, returns testable requirements as JSON |
| Crawler | Visits the site + its sub-pages, returns real locators |
| Developer | Writes the pytest suite |
| Reviewer | Validates it and sends failures back to the Developer |

**Before you run:** add your Gemini key to Colab Secrets (key icon in the left
sidebar) with the name `GOOGLE_API_KEY`, and toggle notebook access on.

## 1. Install

In [ ]:
# --upgrade matters: older SDK builds reject the new AQ. auth keys
!pip install -q --upgrade langchain-google-genai google-genai langgraph playwright pymupdf python-dotenv requests
!playwright install chromium
!playwright install-deps chromium
print("\nInstall complete. Restart runtime only if Colab asks you to.")

## 2. Write the crawler to disk

The crawler runs in a **subprocess**, not inline. Colab already has a running
asyncio event loop, so calling `asyncio.run()` inside a notebook cell throws
`RuntimeError: asyncio.run() cannot be called from a running event loop`.
Shelling out to a separate process side-steps that entirely.

In [ ]:
%%writefile ui_crawler.py
"""
ui_crawler.py  (v2)

WHAT CHANGED FROM v1
--------------------
1. Crawls the base URL *and* its sub-pages, not just the landing page.
   Your SRS talks about /login, /checkboxes, /upload... so the Developer
   agent needs locators from those pages, not just homepage links.

2. The uniqueness check actually works now.
   v1 did:  page.locator("getByRole('button', { name: 'x' })")
   Playwright read that string as a CSS selector, threw an error, and the
   bare `except` returned False. So the good strategies never won and
   everything fell through to the href fallback. Now we build the REAL
   Locator object and count it.

3. Emits Python syntax (get_by_role) instead of JS (getByRole), because
   the generated tests are pytest + Playwright Python.

4. Captures content elements too (headings, flash messages, tables).
   Requirements like "displays the heading 'Welcome to the-internet'"
   are untestable without them.

5. Every element records which page it was found on.

OUTPUT: a flat JSON list, so state['locators'] stays a list.
  {"page": "/login", "tag": "input",
   "locator": "page.get_by_label(\"Username\")", "text": ""}
"""

import asyncio
import json
import os
import sys
from urllib.parse import urljoin, urlparse

from playwright.async_api import async_playwright

# ---------------- CONFIG ----------------

# How many pages to visit in total (the landing page counts as 1).
MAX_PAGES = int(os.getenv("MAX_PAGES", "25"))

# Pages that hang, download files, or pop native auth dialogs.
# Playwright will sit on these forever, so we never visit them.
SKIP_SUBSTRINGS = (
    "/basic_auth",
    "/digest_auth",
    "/download",
    "/redirector",
    "/status_codes/",
    "/nested_frames",
    ".zip",
    ".pdf",
    "mailto:",
)

# Things a user can interact with.
INTERACTIVE_SELECTOR = "a, button, input, select, textarea"

# Things a test needs to assert against.
CONTENT_SELECTOR = "h1, h2, h3, h4, label, #flash, #result, table, .example > p"

# Don't re-collect the 45-link nav menu on all 25 pages.
MAX_ELEMENTS_PER_PAGE = 40


# ---------------- HELPERS ----------------

def py_str(value: str) -> str:
    """
    Turn a Python string into a safe Python string *literal*.
    json.dumps handles the quoting and escaping correctly, and its
    output is valid Python too. Avoids v1's manual backslash juggling.
    """
    return json.dumps(value)


def clean(text) -> str:
    """Collapse whitespace so 'Login\\n ' and 'Login' compare equal."""
    if not text:
        return ""
    return " ".join(text.split()).strip()


async def is_unique(locator) -> bool:
    """
    THE KEY FIX.
    We receive a real Playwright Locator object, not a string, so
    .count() actually queries the DOM instead of throwing.
    """
    try:
        return await locator.count() == 1
    except Exception:
        return False


async def build_locator(page, el, index):
    """
    Try locator strategies best-first. Return the Python code string for
    the first one that matches exactly one element on the page.

    Priority is the order Playwright itself recommends:
      role > label > placeholder > test-id > text > attribute > positional
    """
    tag = await el.evaluate("el => el.tagName.toLowerCase()")
    text = clean(await el.inner_text())
    aria_label = clean(await el.get_attribute("aria-label"))
    placeholder = clean(await el.get_attribute("placeholder"))
    name_attr = await el.get_attribute("name")
    testid = await el.get_attribute("data-testid")
    role_attr = await el.get_attribute("role")
    href = await el.get_attribute("href")
    input_type = (await el.get_attribute("type")) or ""
    el_id = await el.get_attribute("id")

    # Work out the implicit ARIA role from the tag.
    input_roles = {
        "": "textbox",
        "text": "textbox",
        "email": "textbox",
        "password": "textbox",
        "search": "searchbox",
        "checkbox": "checkbox",
        "radio": "radio",
        "submit": "button",
        "button": "button",
        "file": "button",
    }
    role_map = {
        "a": "link",
        "button": "button",
        "select": "combobox",
        "textarea": "textbox",
        "h1": "heading",
        "h2": "heading",
        "h3": "heading",
        "h4": "heading",
        "table": "table",
        "input": input_roles.get(input_type.lower(), "textbox"),
    }
    role = role_attr or role_map.get(tag)

    # ---- 1. ROLE + NAME (most stable, survives redesigns) ----
    if role and text and len(text) < 60:
        code = f"page.get_by_role({py_str(role)}, name={py_str(text)})"
        if await is_unique(page.get_by_role(role, name=text)):
            return code

    # ---- 2. LABEL (best for form fields) ----
    if aria_label:
        code = f"page.get_by_label({py_str(aria_label)})"
        if await is_unique(page.get_by_label(aria_label)):
            return code

    # An <input> is usually labelled by <label for="...">, which
    # get_by_label finds even with no aria-label attribute.
    if tag == "input" and el_id:
        try:
            label_text = clean(
                await page.locator(f'label[for="{el_id}"]').first.inner_text()
            )
            if label_text:
                code = f"page.get_by_label({py_str(label_text)})"
                if await is_unique(page.get_by_label(label_text)):
                    return code
        except Exception:
            pass

    # ---- 3. PLACEHOLDER ----
    if placeholder:
        code = f"page.get_by_placeholder({py_str(placeholder)})"
        if await is_unique(page.get_by_placeholder(placeholder)):
            return code

    # ---- 4. TEST ID ----
    if testid:
        code = f"page.get_by_test_id({py_str(testid)})"
        if await is_unique(page.get_by_test_id(testid)):
            return code

    # ---- 5. EXACT TEXT ----
    if text and len(text) < 50:
        code = f"page.get_by_text({py_str(text)}, exact=True)"
        if await is_unique(page.get_by_text(text, exact=True)):
            return code

    # ---- 6. ATTRIBUTE FALLBACKS ----
    if el_id:
        code = f"page.locator({py_str('#' + el_id)})"
        if await is_unique(page.locator(f"#{el_id}")):
            return code

    if name_attr:
        sel = f'[name="{name_attr}"]'
        code = f"page.locator({py_str(sel)})"
        if await is_unique(page.locator(sel)):
            return code

    if href and tag == "a":
        sel = f'a[href="{href}"]'
        code = f"page.locator({py_str(sel)})"
        if await is_unique(page.locator(sel)):
            return code

    # ---- 7. LAST RESORT (brittle, but better than nothing) ----
    return f"page.locator({py_str(tag)}).nth({index})"


async def scrape_page(page, path, collect_links):
    """Pull every visible, testable element off ONE page."""
    elements = []
    seen = set()

    selector = INTERACTIVE_SELECTOR + ", " + CONTENT_SELECTOR
    if not collect_links:
        # On sub-pages, skip the nav menu; we already captured it.
        selector = selector.replace("a, ", "", 1)

    try:
        found = await page.query_selector_all(selector)
    except Exception as e:
        print(f"  ! query failed on {path}: {e}", file=sys.stderr)
        return elements

    for i, el in enumerate(found):
        if len(elements) >= MAX_ELEMENTS_PER_PAGE:
            break
        try:
            if not await el.is_visible():
                continue

            locator = await build_locator(page, el, i)
            if locator in seen:
                continue
            seen.add(locator)

            tag = await el.evaluate("el => el.tagName.toLowerCase()")
            elements.append({
                "page": path,
                "tag": tag,
                "locator": locator,
                "text": clean(await el.inner_text())[:80],
            })
        except Exception:
            # One bad element shouldn't kill the whole crawl.
            continue

    return elements


async def discover_links(page, base_url):
    """Collect same-site sub-page paths from the landing page."""
    base_host = urlparse(base_url).netloc
    paths = []

    hrefs = await page.eval_on_selector_all(
        "a[href]", "els => els.map(e => e.getAttribute('href'))"
    )

    for href in hrefs:
        if not href or href.startswith("#"):
            continue
        if any(s in href for s in SKIP_SUBSTRINGS):
            continue

        parsed = urlparse(urljoin(base_url, href))

        # Same site only — never wander off to github.com etc.
        if parsed.netloc != base_host:
            continue
        if parsed.path in ("", "/"):
            continue
        if parsed.path not in paths:
            paths.append(parsed.path)

    return paths


# ---------------- MAIN CRAWL ----------------

async def crawl_ui_elements_async(base_url: str):
    all_elements = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,  # v1 used False; headless is far faster for 25 pages
            args=["--disable-blink-features=AutomationControlled"],
        )
        context = await browser.new_context()
        page = await context.new_page()

        # --- Landing page ---
        print(f"[1] Crawling {base_url}", file=sys.stderr)
        await page.goto(base_url, timeout=60000)
        await page.wait_for_load_state("domcontentloaded")
        all_elements += await scrape_page(page, "/", collect_links=True)

        # --- Discover sub-pages ---
        paths = await discover_links(page, base_url)
        paths = paths[: MAX_PAGES - 1]
        print(f"    found {len(paths)} sub-pages to visit", file=sys.stderr)

        # --- Visit each sub-page ---
        for n, path in enumerate(paths, start=2):
            target = urljoin(base_url, path)
            try:
                print(f"[{n}] {path}", file=sys.stderr)
                await page.goto(target, timeout=20000)
                await page.wait_for_load_state("domcontentloaded")
                await page.wait_for_timeout(300)
                all_elements += await scrape_page(page, path, collect_links=False)
            except Exception as e:
                print(f"    ! skipped {path}: {type(e).__name__}", file=sys.stderr)
                continue

        await browser.close()

    pages_hit = len(set(e["page"] for e in all_elements))
    print(
        f"DONE: {len(all_elements)} elements across {pages_hit} pages",
        file=sys.stderr,
    )
    return all_elements


# ---------------- CLI ENTRY ----------------

if __name__ == "__main__":
    if len(sys.argv) > 1:
        result = asyncio.run(crawl_ui_elements_async(sys.argv[1]))
        print(json.dumps(result, indent=2))

In [ ]:
%%writefile ui_runner.py
import asyncio
import json
import sys
from ui_crawler import crawl_ui_elements_async

if __name__ == "__main__":
    url = sys.argv[1]
    result = asyncio.run(crawl_ui_elements_async(url))
    # stdout carries ONLY json; all progress logging goes to stderr
    print(json.dumps(result))

## 3. Config and API key

In [ ]:
import os, json, re, ast, subprocess, sys, asyncio, operator
from datetime import datetime
from typing import TypedDict, List, Annotated

import fitz  # pymupdf
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

# --- API key: Colab Secrets first, .env as fallback ---
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Key loaded from Colab Secrets")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    print("Key loaded from .env")

BASE_URL     = os.getenv("TARGET_URL", "https://the-internet.herokuapp.com")
SRS_PATH     = os.getenv("SRS_PDF_PATH", "Capstone requirements document.docx")
MAX_ATTEMPTS = int(os.getenv("MAX_ATTEMPTS", "4"))
MIN_TESTS    = 30
OUTPUT_DIR   = "agent_outputs"

llm = None
print("Environment ready")

In [ ]:
# Locate the SRS document. Works in Colab and in VS Code / plain Jupyter.
if not os.path.exists(SRS_PATH):
    # try the pdf/ subfolder used by the repo layout
    alt = os.path.join("pdf", SRS_PATH)
    if os.path.exists(alt):
        SRS_PATH = alt
    else:
        try:
            from google.colab import files          # Colab: prompt an upload
            SRS_PATH = list(files.upload().keys())[0]
        except ImportError:                          # local: just report it
            raise FileNotFoundError(
                f"Could not find {SRS_PATH}. Put the SRS next to this notebook "
                "or in a pdf/ subfolder, then re-run this cell."
            )

print("Using SRS:", SRS_PATH)

## 3b. Preflight — check everything before spending API quota

Runs in about 30 seconds and fails loudly if something is wrong, instead of
letting you discover it four minutes into a run.

In [ ]:
import requests

print("PREFLIGHT\n" + "-" * 46)
ok = True

# 1. API key present
key = os.environ.get("GOOGLE_API_KEY", "")
ktype = "Auth key (AQ)" if key.startswith("AQ.") else ("Standard key (AIza)" if key.startswith("AIza") else "unrecognised")
print(f"1. API key           : {'found, ' + ktype if key else 'MISSING'}")
if key.startswith("AIza"):
    print("   NOTE: Standard keys are being retired. Create an auth key if calls fail.")
ok &= bool(key)

# 2. Ask the API which models exist and pick one that works.
#    Hardcoding a model name is the most common cause of an instant failure --
#    names get retired, and the notebook dies on the first call.
MODEL_NAME = "gemini-2.0-flash"
if key:
    try:
        # New-style Auth keys (AQ....) must go in the x-goog-api-key HEADER.
        # The old ?key= query parameter works for legacy AIza keys but can
        # return 401 ACCESS_TOKEN_TYPE_UNSUPPORTED for auth keys, so try the
        # header first and fall back only if that fails.
        url = "https://generativelanguage.googleapis.com/v1beta/models"
        r = requests.get(url, headers={"x-goog-api-key": key}, timeout=30)
        if r.status_code != 200:
            print(f"   header auth returned {r.status_code}, trying ?key=")
            r = requests.get(url, params={"key": key}, timeout=30)
        r.raise_for_status()
        usable = [
            m["name"].replace("models/", "")
            for m in r.json().get("models", [])
            if "generateContent" in m.get("supportedGenerationMethods", [])
        ]
        preferred_names = [
            "gemini-flash-lite-latest",
            "gemini-3.5-flash-lite",
            "gemini-3.1-flash-lite",
            "gemini-flash-latest",
            "gemini-3.7-flash",
            "gemini-3.6-flash",
            "gemini-3.5-flash",
        ]
        preferred = [m for m in preferred_names if m in usable]
        preferred += [m for m in usable if "flash" in m and "thinking" not in m and m not in preferred]
        MODEL_NAME = (preferred or usable)[0]
        print(f"2. Models available  : {len(usable)}")
        print(f"   Selected          : {MODEL_NAME}")
    except Exception as e:
        print(f"2. Model list FAILED : {e}")
        ok = False

    llm = ChatGoogleGenerativeAI(
        model=MODEL_NAME,
        google_api_key=key,
        temperature=0.1,
        request_timeout=45,
        retries=1,
    )

# 3. A tiny smoke call, so auth and quota problems surface now, not later
try:
    reply = llm.invoke([HumanMessage(content="Reply with the word READY only.")])
    print(f"3. LLM smoke test    : {str(reply.content).strip()[:20]}")
except Exception as e:
    print(f"3. LLM smoke test    : FAILED -> {e}")
    ok = False

# 4. SRS readable, and it actually contains requirement IDs
try:
    with fitz.open(SRS_PATH) as doc:
        srs_text = "".join(pg.get_text() for pg in doc)
    n_reqs = len(set(re.findall(r"(?<!N)FR-[A-Z]{1,4}-[0-9]{2}", srs_text)))
    print(f"4. SRS document      : {len(srs_text)} chars, {n_reqs} requirement IDs")
    ok &= n_reqs > 0
except Exception as e:
    print(f"4. SRS document      : FAILED -> {e}")
    ok = False

# 5. Browser installed
try:
    v = subprocess.run([sys.executable, "-m", "playwright", "--version"],
                       capture_output=True, text=True, timeout=60)
    print(f"5. Playwright        : {(v.stdout or v.stderr).strip()}")
except Exception as e:
    print(f"5. Playwright        : FAILED -> {e}")
    ok = False

# 6. Target site reachable
try:
    print(f"6. Target site       : HTTP {requests.get(BASE_URL, timeout=30).status_code}")
except Exception as e:
    print(f"6. Target site       : FAILED -> {e}")
    ok = False

print("-" * 46)
print("ALL CHECKS PASSED - safe to run" if ok else "FIX THE ABOVE BEFORE RUNNING")
if not ok:
    raise RuntimeError("Preflight failed; fix the checks above before running the workflow.")

## 4. Shared state

In [ ]:
class AgentState(TypedDict):
    srs_path: str
    url: str
    requirements: List[dict]
    locators: List[dict]
    generated_code: str
    review_feedback: str
    review_report: dict
    iteration_count: int
    is_code_valid: bool
    # Annotated + operator.add means every node APPENDS here instead of
    # overwriting, so we end up with a full audit trail of the run.
    history: Annotated[List[dict], operator.add]

## 5. Agent A — Extractor

In [ ]:
def llm_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content)


def pdf_extractor_node(state: AgentState):
    print("--- [Extractor] Reading SRS ---")

    full_text = ""
    with fitz.open(state["srs_path"]) as doc:
        for page in doc:
            full_text += page.get_text()
    print(f"    document length: {len(full_text)} chars")

    prompt = f"""
You are a Senior QA Analyst.

Extract every TESTABLE UI REQUIREMENT from the SRS below.

A testable UI requirement is any user-visible behaviour, interaction,
validation, navigation, or element that UI automation can verify.

INCLUDE: page loads, navigation, UI elements, clicks/typing/selects,
validation messages, enabled/disabled and visible/hidden states,
text and heading content, and any edge cases stated in the document.

EXCLUDE: backend logic, database behaviour, vague statements.

RULES:
- Return ONLY a valid JSON array. No markdown, no commentary.
- Each item: {{"id": "FR-FA-02", "description": "...", "path": "/login"}}
- CRITICAL: use the requirement ID EXACTLY as printed in the document
  (FR-G-01, FR-CB-02, FR-FA-02, ...). Do NOT renumber them as FR-001.
  The IDs are how a reviewer traces each test back to the SRS.
- "path" is the URL path the requirement applies to, "/" for the home page.
- One behaviour per item. Do not merge two behaviours into one.
- Do not skip requirements. Do not duplicate them.

SRS DOCUMENT:
{full_text}
"""

    res = llm.invoke([SystemMessage(content="You are a QA expert."),
                      HumanMessage(content=prompt)])
    content = llm_text(res.content)

    start, end = content.find("["), content.rfind("]") + 1
    try:
        requirements = json.loads(content[start:end]) if start != -1 else []
    except json.JSONDecodeError as e:
        print(f"    JSON parse failed: {e}")
        requirements = []

    deduped_requirements = []
    seen_req_ids = set()
    for req in requirements:
        req_id = req.get("id")
        if req_id in seen_req_ids:
            continue
        seen_req_ids.add(req_id)
        deduped_requirements.append(req)
    requirements = deduped_requirements

    # --- traceability check against the SRS itself ---
    # Regex the document for real requirement IDs and compare. This is
    # ground truth: it does not depend on the LLM being honest.
    srs_ids = list(dict.fromkeys(re.findall(r"(?<!N)FR-[A-Z]{1,4}-[0-9]{2}", full_text)))
    got = {r.get("id") for r in requirements}
    missed = [i for i in srs_ids if i not in got]

    print(f"    extracted {len(requirements)} requirements")
    print(f"    SRS contains {len(srs_ids)} functional requirement IDs")
    if missed:
        print(f"    NOT EXTRACTED ({len(missed)}): {', '.join(missed[:12])}"
              + (" ..." if len(missed) > 12 else ""))
    else:
        print("    full SRS coverage")

    return {
        "requirements": requirements,
        "history": [{"agent": "Extractor",
                     "event": f"Extracted {len(requirements)} requirements"}],
    }

## 6. Agent B — Crawler

Runs `ui_runner.py` as a subprocess and parses the JSON it prints.

In [ ]:
def run_crawler_process(url):
    result = subprocess.run(
        [sys.executable, "ui_runner.py", url],
        capture_output=True, text=True,
    )
    print(result.stderr)  # crawl progress
    if result.returncode != 0:
        print("    crawler subprocess failed")
        return []
    out = result.stdout.strip()
    try:
        return json.loads(out[out.find("["):])
    except Exception as e:
        print("    could not parse crawler output:", e)
        return []


async def crawler_node(state: AgentState):
    print(f"--- [Crawler] Scanning {state['url']} ---")
    found = await asyncio.to_thread(run_crawler_process, state["url"])
    pages = len({e["page"] for e in found}) if found else 0
    print(f"    {len(found)} elements across {pages} pages")
    return {
        "locators": found,
        "history": [{"agent": "Crawler",
                     "event": f"Found {len(found)} elements on {pages} pages"}],
    }


def group_locators(locators):
    """Group by page so the Developer knows what exists WHERE."""
    grouped = {}
    for el in locators:
        grouped.setdefault(el["page"], []).append(
            f'  {el["locator"]}   # <{el["tag"]}> {el["text"][:40]}'
        )
    return "\n".join(
        f"PAGE {path}\n" + "\n".join(items)
        for path, items in sorted(grouped.items())
    )

## 7. The validator — deterministic, not an LLM

This is the part that was broken before. Asking an LLM to do compiler-level
checking gave false failures (it flagged `BASE_URL` code as having hardcoded
URLs, and valid `get_by_role` calls as forbidden), so the loop never converged.

Python can check all of this exactly, for free, in milliseconds.

**Design decision:** unknown locators are *reported*, not failed. The old rule
"FAIL if any new selector appears" was impossible to satisfy — the Developer
was simultaneously told to infer locators when one was missing. Two agents
were given contradictory instructions, so every run failed by construction.

In [ ]:
def validate_suite(code: str, requirements: list, min_tests: int = MIN_TESTS):
    """Return (is_valid, report_dict). Pure Python, no LLM."""
    report = {"syntax_error": None, "test_count": 0, "missing_requirements": [],
              "tests_without_assertions": [], "duplicate_tests": [],
              "hardcoded_urls": [], "invalid_playwright_api": [], "unknown_locators": []}

    # 1. Does it even parse?
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        report["syntax_error"] = f"line {e.lineno}: {e.msg}"
        return False, report

    tests = [n for n in ast.walk(tree)
             if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")]
    names = [t.name for t in tests]
    report["test_count"] = len(tests)

    # 2. Duplicate test names (python silently keeps only the last one)
    report["duplicate_tests"] = sorted({n for n in names if names.count(n) > 1})

    # 3. Every requirement covered by at least one test name
    joined = " ".join(names).lower()
    for req in requirements:
        token = req["id"].lower().replace("-", "_")
        if token not in joined:
            report["missing_requirements"].append(req["id"])

    # 4. Every test asserts something
    for t in tests:
        body_text = ast.unparse(t)
        has_plain_assert = any(isinstance(n, ast.Assert) for n in ast.walk(t))
        if "expect(" not in body_text and not has_plain_assert:
            report["tests_without_assertions"].append(t.name)

    # 5. No hardcoded URLs in NAVIGATION.
    #    Only page.goto() counts. A URL inside an assertion is legitimate --
    #    e.g. expect(link).to_have_attribute("href", "http://example.com")
    #    is checking a real attribute value, not hardcoding a target.
    for i, line in enumerate(code.splitlines(), 1):
        if line.strip().startswith("BASE_URL"):
            continue
        if ".goto(" in line and re.search(r"https?://", line):
            report["hardcoded_urls"].append(f"line {i}: {line.strip()[:70]}")

    # 6. Catch JavaScript Playwright assertion names in Python tests.
    invalid_api_pattern = re.compile(r"\.to(?:Be|Have|Contain|Equal|Not)")
    for i, line in enumerate(code.splitlines(), 1):
        if invalid_api_pattern.search(line):
            report["invalid_playwright_api"].append(f"line {i}: {line.strip()[:90]}")

    is_valid = (
        report["syntax_error"] is None
        and report["test_count"] >= min_tests
        and not report["missing_requirements"]
        and not report["tests_without_assertions"]
        and not report["duplicate_tests"]
        and not report["hardcoded_urls"]
        and not report["invalid_playwright_api"]
    )
    return is_valid, report


def report_to_feedback(report, min_tests=MIN_TESTS):
    """Turn the report into instructions the Developer can act on."""
    if report["syntax_error"]:
        return f"The file does not parse. Fix this first: {report['syntax_error']}"

    lines = []
    if report["test_count"] < min_tests:
        lines.append(f"Only {report['test_count']} tests; need at least {min_tests}. ADD the missing ones.")
    if report["missing_requirements"]:
        lines.append("These requirements have no test. Add one test each, "
                     "named test_<id_lowercase_with_underscores>: "
                     + ", ".join(report["missing_requirements"]))
    if report["tests_without_assertions"]:
        lines.append("These tests assert nothing. Add expect() calls: "
                     + ", ".join(report["tests_without_assertions"]))
    if report["duplicate_tests"]:
        lines.append("Duplicate test names (rename or merge): "
                     + ", ".join(report["duplicate_tests"]))
    if report["hardcoded_urls"]:
        lines.append("Replace these hardcoded URLs with f\"{BASE_URL}/path\": "
                     + "; ".join(report["hardcoded_urls"][:5]))
    if report["invalid_playwright_api"]:
        lines.append("These lines use JavaScript Playwright API names. Rewrite them with Python names such as to_be_visible(), to_have_text(), to_contain_text(), to_have_value(): "
                     + "; ".join(report["invalid_playwright_api"][:8]))
    return "\n".join(f"- {l}" for l in lines)

## 8. Agent C — Developer

Generated in **batches**, not one giant response.

Every LLM has a per-response output limit. The previous version asked for the
whole suite at once; at 39 tests it produced 8,640 characters, already close to
the ceiling. At 70 requirements a single response gets cut off mid-function and
the file will not parse. Batching side-steps the limit entirely, and repair
mode regenerates only the handful of tests that failed.

In [ ]:
BATCH_SIZE = 10

HEADER = 'from playwright.sync_api import Page, expect\n\nBASE_URL = "' + BASE_URL + '"\n'


def _clean_code(text: str) -> str:
    """Strip markdown fences and stray import/BASE_URL lines from a chunk."""
    text = str(text).strip()
    text = re.sub(r"^```(?:python)?\s*", "", text)
    text = re.sub(r"```\s*$", "", text).strip()
    keep = [
        ln for ln in text.splitlines()
        if not ln.startswith("from playwright")
        and not ln.startswith("import ")
        and not ln.strip().startswith("BASE_URL")
    ]
    return "\n".join(keep).strip()


def _ask_for_tests(reqs, locators_text, extra_instruction=""):
    """One LLM call -> test functions for this batch of requirements only."""
    prompt = f"""
You are a Senior SDET writing Playwright + pytest tests in Python.

Output ONLY test functions. No imports, no BASE_URL line, no markdown, no prose.
BASE_URL and the imports already exist in the file.

RULES:
- Exactly one test per requirement below.
- Name each test test_<id lowercase, hyphens become underscores>.
  FR-FA-02 becomes: def test_fr_fa_02(page: Page):
- Every test takes "page: Page".
- Navigate with page.goto(f"{{BASE_URL}}{{path}}") using the requirement's path.
  Never write a literal http URL inside a goto call.
- Every test needs at least one Python Playwright expect(...) assertion or a plain Python assert.
- Use Python Playwright assertion method names only: to_be_visible(), to_have_text(), to_contain_text(), to_have_value(). Never use JavaScript names like toBeVisible or toHaveValue.
- For file upload tests, automatically choose the dummy file with page.locator("#file-upload").set_input_files("test_assets/upload_dummy.txt"). Do not require manual file selection.
- Prefer the real locators listed below. If the element you need is not listed,
  write a sensible get_by_role or get_by_text locator yourself - that is allowed
  and expected.

REAL LOCATORS FROM THE LIVE SITE:
{locators_text}

REQUIREMENTS FOR THIS BATCH:
{json.dumps(reqs, indent=1)}
{extra_instruction}
"""
    res = llm.invoke([SystemMessage(content="You are a Senior SDET."),
                      HumanMessage(content=prompt)])
    return _clean_code(llm_text(res.content))


def _drop_tests(code: str, names: list) -> str:
    """Remove named test functions so repaired versions can replace them."""
    for n in names:
        code = re.sub(rf"(?ms)^def {re.escape(n)}\(.*?(?=^def |\Z)", "", code)
    return code.strip()


def playwright_developer_node(state: AgentState):
    attempt = state.get("iteration_count", 0) + 1
    print(f"--- [Developer] attempt {attempt} ---")

    reqs = state["requirements"]
    loc_text = group_locators(state["locators"])
    feedback = state.get("review_feedback", "")
    previous = state.get("generated_code", "")

    # ---------- FIRST PASS: build the whole suite in batches ----------
    if not previous:
        chunks = [HEADER]
        for i in range(0, len(reqs), BATCH_SIZE):
            batch = reqs[i:i + BATCH_SIZE]
            print(f"    batch {i // BATCH_SIZE + 1}: {len(batch)} requirements", end="")
            try:
                chunks.append(_ask_for_tests(batch, loc_text))
                print(" ok")
            except Exception as e:
                print(f" FAILED ({e}) - continuing")
        code_out = "\n\n".join(c for c in chunks if c)

    # ---------- REPAIR PASS: regenerate only what failed ----------
    else:
        ids = list(dict.fromkeys(
            re.findall(r"(?<!N)FR-[A-Z]{1,4}-[0-9]{2}", feedback)))
        broken_names = list(dict.fromkeys(
            re.findall(r"\btest_[a-z0-9_]+", feedback)))
        print(f"    repairing {len(ids)} requirements, {len(broken_names)} named tests")

        # Drop the failing tests, keep everything that passed untouched.
        code_out = _drop_tests(previous, broken_names)

        targets = [r for r in reqs if r["id"] in ids]
        if not targets and broken_names:
            # feedback named tests but not IDs -- map names back to requirements
            wanted = {n.replace("test_", "").replace("_", "").lower()
                      for n in broken_names}
            targets = [r for r in reqs
                       if r["id"].replace("-", "").lower() in wanted]

        for i in range(0, len(targets), BATCH_SIZE):
            batch = targets[i:i + BATCH_SIZE]
            try:
                fixed = _ask_for_tests(
                    batch, loc_text,
                    extra_instruction="\nFIX THESE PROBLEMS:\n" + feedback + "\n",
                )
                code_out += "\n\n" + fixed
            except Exception as e:
                print(f"    repair batch failed: {e}")

    print(f"    suite is now {len(code_out)} chars")
    return {
        "generated_code": code_out,
        "history": [{"agent": "Developer", "attempt": attempt,
                     "chars": len(code_out)}],
    }

## 9. Agent D — Reviewer

In [ ]:
def reviewer_node(state: AgentState):
    print("--- [Reviewer] Validating ---")

    is_valid, report = validate_suite(state["generated_code"], state["requirements"])

    print(f"    tests: {report['test_count']}")
    print(f"    syntax: {'OK' if not report['syntax_error'] else report['syntax_error']}")
    print(f"    uncovered requirements: {len(report['missing_requirements'])}")
    print(f"    tests missing assertions: {len(report['tests_without_assertions'])}")
    print(f"    hardcoded URLs: {len(report['hardcoded_urls'])}")
    print(f"    invalid Playwright APIs: {len(report['invalid_playwright_api'])}")
    print(f"    RESULT: {'PASSED' if is_valid else 'FAILED'}")

    feedback = "" if is_valid else report_to_feedback(report)

    return {
        "review_feedback": feedback,
        "review_report": report,
        "is_code_valid": is_valid,
        "iteration_count": state.get("iteration_count", 0) + 1,
        "history": [{"agent": "Reviewer", "valid": is_valid, "report": report}],
    }

## 10. Wire the graph

In [ ]:
def router(state: AgentState):
    if state.get("is_code_valid"):
        print("Workflow complete: suite passed validation.")
        return "end"
    if state.get("iteration_count", 0) >= MAX_ATTEMPTS:
        print(f"Stopping: hit {MAX_ATTEMPTS} attempts without passing.")
        return "end"
    print("Sending failures back to the Developer.\n")
    return "fix"


workflow = StateGraph(AgentState)
workflow.add_node("extractor", pdf_extractor_node)
workflow.add_node("crawler", crawler_node)
workflow.add_node("developer", playwright_developer_node)
workflow.add_node("reviewer", reviewer_node)

workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "crawler")
workflow.add_edge("crawler", "developer")
workflow.add_edge("developer", "reviewer")
workflow.add_conditional_edges("reviewer", router, {"fix": "developer", "end": END})

app = workflow.compile()
print("Graph compiled")

## 11. Run

In [ ]:
inputs = {
    "srs_path": SRS_PATH,
    "url": BASE_URL,
    "iteration_count": 0,
    "history": [],
    "is_code_valid": False,
    "review_feedback": "",
    "generated_code": "",
}

print("Starting agentic workflow...\n")
final_output = await app.ainvoke(inputs)
print("\nDone. Valid:", final_output["is_code_valid"])

## 12. Save the outputs

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

py_path  = f"{OUTPUT_DIR}/test_generated_{run_id}.py"
log_path = f"{OUTPUT_DIR}/full_log_{run_id}.json"
txt_path = f"{OUTPUT_DIR}/execution_log_{run_id}.txt"

with open(py_path, "w", encoding="utf-8") as f:
    f.write(final_output.get("generated_code", "# nothing generated"))

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, default=str)

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("=== AGENTIC WORKFLOW EXECUTION LOG ===\n")
    f.write(f"Timestamp: {datetime.now()}\n")
    f.write(f"Final status: {'PASSED' if final_output['is_code_valid'] else 'FAILED'}\n")
    f.write(f"Attempts: {final_output.get('iteration_count')}\n")
    f.write("-" * 50 + "\n\n")
    for entry in final_output.get("history", []):
        f.write(f"[{entry.get('agent')}]\n")
        for k, v in entry.items():
            if k != "agent":
                f.write(f"  {k}: {v}\n")
        f.write("\n")

print("Saved:")
for p in (py_path, log_path, txt_path):
    print(" ", p)

## 13. Actually run the generated tests

Validation proves the file is *well-formed*. Running it proves the tests are
*correct*. This is the real scoreboard for your capstone — a pass rate is a
much stronger result than "the reviewer agent approved it".

In [ ]:
pytest_args = [sys.executable, "-m", "pytest", py_path, "--browser", "chromium", "-q"]
if os.getenv("HEADLESS", "true").lower() in {"0", "false", "no"}:
    pytest_args.append("--headed")

pytest_result = subprocess.run(
    pytest_args,
    capture_output=True,
    text=True,
)
pytest_output = (pytest_result.stdout or "") + (pytest_result.stderr or "")
print("\n".join(pytest_output.splitlines()[-40:]))
print(f"pytest exit code: {pytest_result.returncode}")